# 第 09 课｜NumPy 数组与张量基础

**学习目标**：把第 08 课用列表和循环实现的向量、矩阵运算，转换为 NumPy 数组运算；能够读懂 `shape`、`ndim`、`dtype`、索引、`reshape`、`@`、广播和 `axis`，最后重现 `QKᵀ / √D`。

- **预计用时**：2–3 小时（包含练习）。
- **先修知识**：第 08 课的向量、点积与矩阵形状，以及基础 Python 函数、列表和 `assert`。
- **环境**：Python 3（建议 3.10 以上）、NumPy、Jupyter；**不需要 GPU 或 PyTorch**。
- **学习方式**：先预测形状和输出，再运行、修改、解释；练习做完后再看答案。

> 这里说的“张量”主要指可用多维数组表示的数据。NumPy 的 `ndarray` 可以表示标量、向量、矩阵以及更多维数组；真正的深度学习训练还需要自动微分，之后会用 PyTorch 学习。

## 0. 安装与检查

在**终端**中运行（只需安装一次）：

```bash
python -m pip install numpy jupyterlab ipykernel
```

如果你已经安装并启动了 Jupyter，先检查 Notebook 的 Python 内核是否能导入 NumPy。出现 `ModuleNotFoundError` 时，请把 NumPy 安装到该内核使用的 Python 环境里，必要时重启内核。下面这个单元不会联网、安装软件或修改环境。

In [1]:
import numpy as np
print("NumPy 版本：", np.__version__)
print("安装检查通过。")

NumPy 版本： 2.4.2
安装检查通过。


## 1. Python 列表与 NumPy 数组：`+` 的含义不同

第 08 课用 `for` 循环来做向量加法；因为 Python 列表的 `+` 是**拼接**。NumPy 数组的 `+` 才会在对应位置相加。

**先猜**：下面最后两行分别输出什么？

In [2]:
list_a = [1, 2, 3]
list_b = [4, 5, 6]
print("Python 列表 +：", list_a + list_b)

a = np.array(list_a)
b = np.array(list_b)
print("NumPy 数组 +：", a + b)
print("NumPy 数组 *：", a * b)

Python 列表 +： [1, 2, 3, 4, 5, 6]
NumPy 数组 +： [5 7 9]
NumPy 数组 *： [ 4 10 18]


`a * b` 也是**逐元素相乘**，不是点积；点积使用 `a @ b`，它把对应元素相乘后求和。请回想第 08 课为什么两个向量必须具有兼容形状。

In [3]:
print("a * b（逐元素，仍是向量）：", a * b)
print("a @ b（点积，结果是数）：", a @ b)
print("手算核对：", 1 * 4 + 2 * 5 + 3 * 6)
assert a @ b == 32

a * b（逐元素，仍是向量）： [ 4 10 18]
a @ b（点积，结果是数）： 32
手算核对： 32


## 2. `ndim`、`shape`、`size`、`dtype` 是什么？

- `ndim`：数组有几条**轴**，即维度数。矩阵的 `ndim` 是 `2`。
- `shape`：每条轴各有多少个元素，形式是元组。例如 `(2, 3)` 表示 2 行 3 列。
- `size`：元素总数。
- `dtype`：每个元素的数据类型，例如 `int64` 或 `float32`。具体默认整数位宽可能随平台不同。

注意：数学里说“三维向量”通常指 **3 个分量**；NumPy 的“一维数组”指它只有 **1 条轴**。`[1, 2, 3]` 是 3 分量向量，但 `ndim == 1`。

In [4]:
scalar = np.array(7)
vector = np.array([7, 8, 9])
matrix = np.array([[1, 2, 3], [4, 5, 6]])
batch = np.arange(24).reshape(2, 3, 4)

for name, arr in [("标量", scalar), ("向量", vector), ("矩阵", matrix), ("三维数组", batch)]:
    print(f"{name:5s}  ndim={arr.ndim}  shape={arr.shape}  size={arr.size}  dtype={arr.dtype}")

assert vector.ndim == 1 and vector.shape == (3,)
assert matrix.shape == (2, 3)
assert batch.shape == (2, 3, 4)

标量     ndim=0  shape=()  size=1  dtype=int64
向量     ndim=1  shape=(3,)  size=3  dtype=int64
矩阵     ndim=2  shape=(2, 3)  size=6  dtype=int64
三维数组   ndim=3  shape=(2, 3, 4)  size=24  dtype=int64


### 模型里的 `(B, T, D)`

这三个字母以后会反复出现：

| 轴位置 | 符号 | 意义 |
|---|---|---|
| `axis=0` | `B` | batch：一次处理的样本数 |
| `axis=1` | `T` | sequence length：每条序列的位置数 |
| `axis=2` | `D` | hidden size：每个位置的特征数 |

下面用连续整数填充一个形状为 `(2, 3, 4)` 的数组，只是为了学习**布局与运算**，不是训练好的 embedding。

In [5]:
hidden = np.arange(2 * 3 * 4).reshape(2, 3, 4)
print("hidden.shape:", hidden.shape)
print("第一个样本：\n", hidden[0])
print("第一个样本的第二个位置向量：", hidden[0, 1])
print("这个向量的 shape：", hidden[0, 1].shape)

hidden.shape: (2, 3, 4)
第一个样本：
 [[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]]
第一个样本的第二个位置向量： [4 5 6 7]
这个向量的 shape： (4,)


## 3. `dtype`：数值的存储类型

模型工程中经常见到 `float32`、`float16`、`bfloat16` 等类型。今天先使用 `float64` / `float32` 理解：类型影响数字的表示方式、存储和计算精度。`astype()` 可以创建转换类型后的数组。

为什么这重要？**内存近似等于元素数量 × 每个元素字节数**（这里只算数组数据，不算对象开销）。稍后学习显存和量化时会再次用到。

In [6]:
x64 = np.array([0.1, 0.2, 0.3], dtype=np.float64)
x32 = x64.astype(np.float32)
print("float64:", x64.dtype, x64.nbytes, "字节")
print("float32:", x32.dtype, x32.nbytes, "字节")
print("转换后原数组不变：", x64.dtype)
assert x64.nbytes == 24 and x32.nbytes == 12

float64: float64 24 字节
float32: float32 12 字节
转换后原数组不变： float64


## 4. 索引与切片：明确保留了哪些轴

- `m[0, 1]`：第 0 行第 1 列，得到一个元素。
- `m[0, :]`：第 0 行的全部列，得到一维数组。
- `m[:, 1]`：所有行的第 1 列，得到一维数组。
- `m[0:2, 1:3]`：按行、列分别切片，得到二维数组。

切片的结束索引**不包含**在内。

In [8]:
m = np.array([[10, 20, 30], [40, 50, 60]])
print("m.shape：", m.shape)
print("m[0, 1]：", m[0, 1])
print("m[0, :]：", m[0, :], "shape：", m[0, :].shape)
print("m[:, 1]：", m[:, 1], "shape：", m[:, 1].shape)
print("m[0:2, 1:3]：\n", m[0:2, 1:3], "shape：", m[0:2, 1:3].shape)
assert m[:, 1].shape == (2,)

m.shape： (2, 3)
m[0, 1]： 20
m[0, :]： [10 20 30] shape： (3,)
m[:, 1]： [20 50] shape： (2,)
m[0:2, 1:3]：
 [[20 30]
 [50 60]] shape： (2, 2)


## 5. `reshape`：换形状，不换元素总数

`np.arange(12)` 生成 `0` 到 `11` 的 12 个数。可以把它整理成 `(3, 4)` 或 `(2, 2, 3)`；不能整理成 `(5, 3)`，因为元素数量不匹配。`reshape` 改变的是如何看待这些元素的轴，不是凭空产生元素。

实践提示：某些 `reshape` 结果可能与原数组共享底层数据；不要想当然地认为得到的总是独立副本，需要独立副本时用 `.copy()`。

In [9]:
flat = np.arange(12)
reshaped_2d = flat.reshape(3, 4)
reshaped_3d = flat.reshape(2, 2, 3)
print("flat：", flat.shape)
print("二维：", reshaped_2d.shape, "\n", reshaped_2d)
print("三维：", reshaped_3d.shape)
try:
    flat.reshape(5, 3)
except ValueError as error:
    print("故意触发的 reshape 错误：", str(error).splitlines()[0])

flat： (12,)
二维： (3, 4) 
 [[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]]
三维： (2, 2, 3)
故意触发的 reshape 错误： cannot reshape array of size 12 into shape (5,3)


### 一个常见混淆：`(3,)` 与 `(3, 1)` 不一样

一维数组的 `.T` 不会让它变成列向量；先 `reshape(3, 1)` 才得到真正的二维列数组。

In [10]:
v = np.array([1, 2, 3])
column = v.reshape(3, 1)
print("v.shape：", v.shape)
print("v.T.shape：", v.T.shape)
print("column.shape：", column.shape)
print("column.T.shape：", column.T.shape)
assert v.T.shape == (3,)
assert column.T.shape == (1, 3)

v.shape： (3,)
v.T.shape： (3,)
column.shape： (3, 1)
column.T.shape： (1, 3)


## 6. `*` 与 `@`：逐元素乘法和矩阵乘法

对二维数组：

- `A * B`：同形状时对应位置相乘（也可能发生广播）。
- `A @ B`：矩阵乘法；内侧维度必须匹配。

形状规则：

$
(m,n)\ @\ (n,p)\ \longrightarrow\ (m,p)
$

例如 `(2, 3) @ (3, 2)` 得到 `(2, 2)`。试着先预测实际数值，再运行。

In [11]:
A = np.array([[1, 2, 3], [4, 5, 6]])  # (2, 3)
B = np.array([[1, 0], [0, 1], [1, 1]])  # (3, 2)
product = A @ B
print("A.shape：", A.shape, "B.shape：", B.shape)
print("A @ B：\n", product)
print("结果形状：", product.shape)
assert product.shape == (2, 2)
np.testing.assert_array_equal(product, [[4, 5], [10, 11]])

A.shape： (2, 3) B.shape： (3, 2)
A @ B：
 [[ 4  5]
 [10 11]]
结果形状： (2, 2)


**矩阵乘向量**的形状是 `(m,n) @ (n,) → (m,)`。下面与第 08 课的手算结果对齐。

In [12]:
M = np.array([[1, 2, 0], [0, 1, 3]])
x = np.array([2, 1, 1])
print("M @ x：", M @ x)
assert (M @ x).tolist() == [4, 4]
try:
    M @ np.array([1, 2])
except ValueError as error:
    print("矩阵乘法维度错误：", str(error).splitlines()[0])

M @ x： [4 4]
矩阵乘法维度错误： matmul: Input operand 1 has a mismatch in its core dimension 0, with gufunc signature (n?,k),(k,m?)->(n?,m?) (size 2 is different from 3)


## 7. Broadcasting（广播）：让形状兼容的数组共同运算

假设每个 token 的向量都要加同一个偏置（bias）：

- hidden 的形状：`(B, T, D) = (2, 3, 4)`；
- bias 的形状：`(D,) = (4,)`。

NumPy 会从**末尾的轴**开始比较维度：两边相同，或其中一边为 `1`，就能按规则广播。结果形状是 `(2, 3, 4)`。

广播是便利工具，也可能让本该报错的形状意外通过；**先看 shape，再看数值**。

In [13]:
hidden = np.arange(24).reshape(2, 3, 4)
bias = np.array([10, 20, 30, 40])
updated = hidden + bias
print("hidden.shape：", hidden.shape)
print("bias.shape：", bias.shape)
print("updated.shape：", updated.shape)
print("原来第一个位置：", hidden[0, 0])
print("加 bias 后：", updated[0, 0])
assert updated.shape == (2, 3, 4)
np.testing.assert_array_equal(updated[0, 0], [10, 21, 32, 43])

hidden.shape： (2, 3, 4)
bias.shape： (4,)
updated.shape： (2, 3, 4)
原来第一个位置： [0 1 2 3]
加 bias 后： [10 21 32 43]


In [14]:
try:
    np.ones((2, 3)) + np.ones((4,))
except ValueError as error:
    print("不能广播的形状：", str(error).splitlines()[0])

不能广播的形状： operands could not be broadcast together with shapes (2,3) (4,) 


## 8. `axis`：沿哪个方向汇总？

对形状 `(B, T, D)` 的数组：

- `sum(axis=2)`：对最后一个特征轴求和，输出 `(B, T)`；
- `sum(axis=1)`：对序列位置求和，输出 `(B, D)`；
- `sum(axis=0)`：对 batch 求和，输出 `(T, D)`；
- `keepdims=True`：被汇总的轴仍保留，只是长度变为 `1`。

`axis=-1` 表示最后一条轴，后续 softmax 常使用这个写法。

In [15]:
cube = np.arange(24).reshape(2, 3, 4)
print("输入：", cube.shape)
for axis in [0, 1, 2, -1]:
    print(f"sum(axis={axis})：", cube.sum(axis=axis).shape)
print("sum(axis=-1, keepdims=True)：", cube.sum(axis=-1, keepdims=True).shape)
print("第一个样本在最后一维求和：", cube.sum(axis=-1)[0])
assert cube.sum(axis=-1).shape == (2, 3)

输入： (2, 3, 4)
sum(axis=0)： (3, 4)
sum(axis=1)： (2, 4)
sum(axis=2)： (2, 3)
sum(axis=-1)： (2, 3)
sum(axis=-1, keepdims=True)： (2, 3, 1)
第一个样本在最后一维求和： [ 6 22 38]


## 9. 连接到 Transformer：真正用数组计算 `QKᵀ/√D`

第 08 课通过两层循环，为每个 Query 和 Key 做点积。NumPy 可以一次写成矩阵运算：

$
S=\frac{QK^\mathsf{T}}{\sqrt{D}}
$

其中：

- `Q.shape = (Tq, D)`，共有 `Tq` 个查询；
- `K.shape = (Tk, D)`，共有 `Tk` 个键；
- `K.T.shape = (D, Tk)`；
- `S.shape = (Tq, Tk)`。

注意 `Tq` 和 `Tk` **可以不同**，不一定生成方阵。今天只算缩放后的匹配分数；因果掩码、softmax 和对 `V` 加权还没有加入。

In [16]:
Q = np.array([[1, 0], [0, 1], [1, 1]], dtype=np.float64)
K = np.array([[1, 0], [0, 1], [1, 1]], dtype=np.float64)
raw_scores = Q @ K.T
scaled_scores = raw_scores / np.sqrt(Q.shape[1])
print("Q.shape：", Q.shape)
print("K.T.shape：", K.T.shape)
print("Q @ K.T（未缩放）：\n", raw_scores)
print("除以 sqrt(D) 后：\n", np.round(scaled_scores, 4))
assert scaled_scores.shape == (3, 3)

Q.shape： (3, 2)
K.T.shape： (2, 3)
Q @ K.T（未缩放）：
 [[1. 0. 1.]
 [0. 1. 1.]
 [1. 1. 2.]]
除以 sqrt(D) 后：
 [[0.7071 0.     0.7071]
 [0.     0.7071 0.7071]
 [0.7071 0.7071 1.4142]]


把这段运算写成函数，并**显式检查**形状：这样输入错误时，我们能给出易懂的报错。下面的函数与 `.py` 里对应实现相同。

In [17]:
# 以下函数与配套 .py 脚本的实现相同。
def dot_numpy(a, b):
    """两个一维向量的点积：返回 Python float。"""
    left = np.asarray(a, dtype=np.float64)
    right = np.asarray(b, dtype=np.float64)
    if left.ndim != 1 or right.ndim != 1:
        raise ValueError("点积要求两个输入都是一维向量")
    if left.size == 0 or left.shape != right.shape:
        raise ValueError("点积要求非空且长度相同的一维向量")
    return float(left @ right)


def cosine_similarity_numpy(a, b):
    """计算非零一维向量的余弦相似度。"""
    left = np.asarray(a, dtype=np.float64)
    right = np.asarray(b, dtype=np.float64)
    # 先复用点积函数检查输入是否合法。
    numerator = dot_numpy(left, right)
    denominator = float(np.linalg.norm(left) * np.linalg.norm(right))
    if denominator == 0:
        raise ValueError("零向量无法计算余弦相似度")
    return numerator / denominator


def matrix_vector_numpy(matrix, vector):
    """二维矩阵 @ 一维向量，得到一维向量。"""
    mat = np.asarray(matrix, dtype=np.float64)
    vec = np.asarray(vector, dtype=np.float64)
    if mat.ndim != 2 or vec.ndim != 1:
        raise ValueError("矩阵乘向量要求二维矩阵和一维向量")
    if 0 in mat.shape or vec.size == 0 or mat.shape[1] != vec.shape[0]:
        raise ValueError("矩阵和向量的形状不匹配，或输入为空")
    return mat @ vec


def scaled_qk_scores(queries, keys):
    """计算二维 Q @ K.T / sqrt(D)，输出尚未归一化的注意力分数。

    Q 的形状：(Tq, D)；K 的形状：(Tk, D)。
    输出形状：(Tq, Tk)。
    """
    q = np.asarray(queries, dtype=np.float64)
    k = np.asarray(keys, dtype=np.float64)
    if q.ndim != 2 or k.ndim != 2:
        raise ValueError("Q 和 K 必须都是二维数组")
    if 0 in q.shape or 0 in k.shape:
        raise ValueError("Q 和 K 的维度都必须大于 0")
    if q.shape[1] != k.shape[1]:
        raise ValueError("Q 和 K 的向量维度 D 必须相同")
    d_k = q.shape[1]
    return (q @ k.T) / np.sqrt(d_k)


def batched_scaled_qk_scores(queries, keys):
    """批量计算 QKᵀ/√D；本课程要求两侧批次数 B 完全相同。

    Q：(B, Tq, D)；K：(B, Tk, D)；返回：(B, Tq, Tk)。
    """
    q = np.asarray(queries, dtype=np.float64)
    k = np.asarray(keys, dtype=np.float64)
    if q.ndim != 3 or k.ndim != 3:
        raise ValueError("批量 Q 和 K 都必须是三维数组")
    if 0 in q.shape or 0 in k.shape:
        raise ValueError("所有维度必须大于 0")
    if q.shape[0] != k.shape[0]:
        raise ValueError("两个数组的 batch 大小必须相同")
    if q.shape[2] != k.shape[2]:
        raise ValueError("两个数组的向量维度 D 必须相同")
    d_k = q.shape[2]
    return (q @ np.swapaxes(k, -1, -2)) / np.sqrt(d_k)

In [18]:
check = scaled_qk_scores(Q, K)
np.testing.assert_allclose(check, scaled_scores)
print("手写函数与矩阵表达式结果一致：", check.shape)

# 查询 2 个位置、键 4 个位置，得到 (2, 4)：
Q_rect = np.ones((2, 3))
K_rect = np.ones((4, 3))
print("非方阵分数形状：", scaled_qk_scores(Q_rect, K_rect).shape)
assert scaled_qk_scores(Q_rect, K_rect).shape == (2, 4)

手写函数与矩阵表达式结果一致： (3, 3)
非方阵分数形状： (2, 4)


## 10. 进阶观察：批量 `QKᵀ` 为什么不是 `K.T`？

真实模型往往带有 batch 轴。假设 `Q.shape=(B,Tq,D)`，`K.shape=(B,Tk,D)`，需要把 **K 的最后两条轴交换**，而不是用普通二维转置的直觉处理整个三维数组。

`np.swapaxes(K, -1, -2)` 把 `(..., Tk, D)` 转为 `(..., D, Tk)`，再用 `@` 得到 `(..., Tq, Tk)`。

这里暂不涉及多注意力头，后续会从 `(B,T,D)` 进一步拆成 `(B,H,T,d_head)`。

In [19]:
Q_batch = np.stack([Q, 2 * Q])  # (2, 3, 2)
K_batch = np.stack([K, K])      # (2, 3, 2)
print("Q_batch.shape：", Q_batch.shape)
print("K_batch.shape：", K_batch.shape)
print("批量 K 交换末尾两轴：", np.swapaxes(K_batch, -1, -2).shape)
batch_scores = batched_scaled_qk_scores(Q_batch, K_batch)
print("批量分数形状：", batch_scores.shape)
np.testing.assert_allclose(batch_scores[0], scaled_scores)
np.testing.assert_allclose(batch_scores[1], 2 * scaled_scores)

Q_batch.shape： (2, 3, 2)
K_batch.shape： (2, 3, 2)
批量 K 交换末尾两轴： (2, 2, 3)
批量分数形状： (2, 3, 3)


---

## 11. 自己动手：先做，再往下看答案

**练习 A：形状推断**（约 10 分钟）。先在纸上写结果，再运行下面的单元。请写出每行的 `shape`，并解释哪条轴被保留、去掉或合并。

In [20]:
exercise_a = np.arange(24).reshape(2, 3, 4)
print("A1：", exercise_a[:, 1, :].shape)
print("A2：", exercise_a[0, :, 2].shape)
print("A3：", exercise_a.sum(axis=1).shape)
print("A4：", exercise_a.reshape(6, 4).shape)
print("A5：", exercise_a.mean(axis=-1, keepdims=True).shape)

A1： (2, 4)
A2： (3,)
A3： (2, 4)
A4： (6, 4)
A5： (2, 3, 1)


**练习 B：实现余弦相似度**（约 15 分钟）。利用 `@` 与 `np.linalg.norm`，编写 `cosine_similarity_ex(a, b)`。要求：

1. 输入必须为非空且等长的一维向量。
2. 任何一个是零向量时抛出 `ValueError`。
3. 对 `[3,4]` 和 `[3,0]`，返回 `0.6`（允许微小浮点误差）。

函数下面的检测单元在你还没实现时会提示“待完成”，不影响其他 Notebook 单元运行。

In [ ]:
def cosine_similarity_ex(a, b):
    # TODO：请在这里实现；建议先看 a.shape、b.shape、a @ b。
    raise NotImplementedError("练习 B 尚未完成")

In [ ]:
try:
    result_b = cosine_similarity_ex([3, 4], [3, 0])
    np.testing.assert_allclose(result_b, 0.6)
    print("练习 B 通过：", result_b)
except NotImplementedError:
    print("练习 B 待完成：先实现上一个单元的函数，再重新运行")

**练习 C：自己写 `QKᵀ/√D`**（约 20 分钟）。实现 `scaled_qk_ex(q, k)`，要求：

- 不使用 `for` 循环；
- 检查 `q`、`k` 是二维、非空，最后一维一样；
- `q.shape=(2,3)`、`k.shape=(4,3)` 时结果应为 `(2,4)`；
- 和第 08 课相同的小例子应该得到相同的结果。

In [ ]:
def scaled_qk_ex(q, k):
    # TODO：使用 np.asarray(..., dtype=np.float64)、@、.T、np.sqrt。
    raise NotImplementedError("练习 C 尚未完成")

In [ ]:
try:
    result_c = scaled_qk_ex(Q, K)
    np.testing.assert_allclose(result_c, scaled_scores)
    assert scaled_qk_ex(np.ones((2, 3)), np.ones((4, 3))).shape == (2, 4)
    print("练习 C 通过：", result_c.shape)
except NotImplementedError:
    print("练习 C 待完成：先实现上一个单元的函数，再重新运行")

**可选练习 D（进阶）**：比较 `K_batch.T` 与 `np.swapaxes(K_batch, -1, -2)` 的形状，解释为什么在三维数组上不能直接沿用二维 `.T` 的想法。

In [ ]:
print("K_batch.T.shape：", K_batch.T.shape)
print("swapaxes(K_batch, -1, -2).shape：", np.swapaxes(K_batch, -1, -2).shape)
print("想一想：第一个结果的 batch 轴跑到哪里去了？")

---

## 12. 参考答案（建议做完练习后再阅读）

练习 A 的形状应该依次为 `(2,4)`、`(3,)`、`(2,4)`、`(6,4)`、`(2,3,1)`。

练习 B / C 的完整参考实现如下。它们使用 `_answer` 后缀，不会覆盖你在练习区写的函数。

In [21]:
def cosine_similarity_answer(a, b):
    left = np.asarray(a, dtype=np.float64)
    right = np.asarray(b, dtype=np.float64)
    if left.ndim != 1 or right.ndim != 1 or left.size == 0 or left.shape != right.shape:
        raise ValueError("输入必须是非空、等长的一维向量")
    denominator = np.linalg.norm(left) * np.linalg.norm(right)
    if denominator == 0:
        raise ValueError("零向量无法计算余弦相似度")
    return float((left @ right) / denominator)

np.testing.assert_allclose(cosine_similarity_answer([3, 4], [3, 0]), 0.6)
print("参考答案 B：", cosine_similarity_answer([3, 4], [3, 0]))

参考答案 B： 0.6


In [22]:
def scaled_qk_answer(q, k):
    q = np.asarray(q, dtype=np.float64)
    k = np.asarray(k, dtype=np.float64)
    if q.ndim != 2 or k.ndim != 2:
        raise ValueError("Q 和 K 必须为二维")
    if 0 in q.shape or 0 in k.shape or q.shape[1] != k.shape[1]:
        raise ValueError("Q 和 K 的形状不匹配或输入为空")
    return (q @ k.T) / np.sqrt(q.shape[1])

np.testing.assert_allclose(scaled_qk_answer(Q, K), scaled_scores)
assert scaled_qk_answer(np.ones((2, 3)), np.ones((4, 3))).shape == (2, 4)
print("参考答案 C：\n", np.round(scaled_qk_answer(Q, K), 4))

参考答案 C：
 [[0.7071 0.     0.7071]
 [0.     0.7071 0.7071]
 [0.7071 0.7071 1.4142]]


## 13. 自动验收：将本课计算与已知答案核对

这组固定检查包括向量、矩阵、广播、形状、缩放注意力和错误输入。所有用到的函数都已在本 Notebook 中定义。Notebook 内的测试和配套 `.py` 脚本使用**同一组检查代码**。

In [23]:
def run_checks():
    """独立的固定案例测试；不使用随机数，也不需要 GPU。"""
    passed = 0

    def check_close(label, actual, expected):
        nonlocal passed
        try:
            np.testing.assert_allclose(actual, expected, rtol=1e-12, atol=1e-12)
        except AssertionError as exc:
            raise AssertionError(f"检查失败：{label}") from exc
        passed += 1

    def check_equal(label, actual, expected):
        nonlocal passed
        if actual != expected:
            raise AssertionError(f"检查失败：{label}；预期 {expected!r}；实际 {actual!r}")
        passed += 1

    def expect_value_error(label, fn):
        nonlocal passed
        try:
            fn()
        except ValueError:
            passed += 1
        else:
            raise AssertionError(f"检查失败：{label} 应抛出 ValueError")

    a = np.array([1, 2, 3])
    b = np.array([-1, 0, 2])
    matrix = np.array([[1, 2, 0], [0, 1, 3]])
    cube = np.arange(24).reshape(2, 3, 4)
    q = np.array([[1, 0], [0, 1], [1, 1]])
    k = q.copy()
    expected_raw = np.array([[1, 0, 1], [0, 1, 1], [1, 1, 2]])
    scores = scaled_qk_scores(q, k)

    check_close("NumPy 数组逐元素加法", a + b, [0, 2, 5])
    check_close("数组逐元素乘法", a * b, [-1, 0, 6])
    check_equal("向量形状", a.shape, (3,))
    check_equal("矩阵形状", matrix.shape, (2, 3))
    check_equal("三维数组形状", cube.shape, (2, 3, 4))
    check_equal("三维数组维度数", cube.ndim, 3)
    check_equal("三维数组元素数", cube.size, 24)
    check_equal("reshape 后形状", np.arange(12).reshape(3, 4).shape, (3, 4))
    check_close("二维索引", matrix[0, 1], 2)
    check_close("取整列", matrix[:, 1], [2, 1])
    check_close("矩阵求和 axis=2", cube.sum(axis=2)[0], [6, 22, 38])
    check_equal("矩阵求和 axis=2 形状", cube.sum(axis=2).shape, (2, 3))
    check_equal("保留求和轴的形状", cube.sum(axis=2, keepdims=True).shape, (2, 3, 1))
    check_equal("广播结果形状", (cube + np.ones(4)).shape, (2, 3, 4))
    check_close("向量点积", dot_numpy(a, b), 5)
    check_close("垂直向量余弦相似度", cosine_similarity_numpy([1, 0], [0, 1]), 0)
    check_close("方向相同余弦相似度", cosine_similarity_numpy([2, 0], [1, 0]), 1)
    check_close("3-4-5 向量余弦相似度", cosine_similarity_numpy([3, 4], [3, 0]), 0.6)
    check_close("矩阵乘向量", matrix_vector_numpy(matrix, [2, 1, 1]), [4, 4])
    check_equal("QK 分数矩阵形状", scores.shape, (3, 3))
    check_close("QK 分数矩阵数值", scores, expected_raw / np.sqrt(2))
    check_equal("非方形 QK 分数矩阵形状", scaled_qk_scores(np.ones((2, 3)), np.ones((4, 3))).shape, (2, 4))
    q_batch = np.stack([q, q * 2])
    k_batch = np.stack([k, k])
    batched_scores = batched_scaled_qk_scores(q_batch, k_batch)
    check_equal("批量 QK 分数形状", batched_scores.shape, (2, 3, 3))
    check_close("批量第一个样本", batched_scores[0], scores)
    check_close("批量第二个样本", batched_scores[1], scores * 2)

    expect_value_error("点积维度不同", lambda: dot_numpy([1, 2], [3]))
    expect_value_error("点积输入非一维", lambda: dot_numpy([[1, 2]], [1, 2]))
    expect_value_error("零向量余弦", lambda: cosine_similarity_numpy([0, 0], [1, 0]))
    expect_value_error("矩阵乘向量形状不同", lambda: matrix_vector_numpy(matrix, [1, 2]))
    expect_value_error("QK 输入不是二维", lambda: scaled_qk_scores([1, 2], [[1, 2]]))
    expect_value_error("QK 向量维度不同", lambda: scaled_qk_scores([[1, 2]], [[1, 2, 3]]))
    expect_value_error("QK 空输入", lambda: scaled_qk_scores(np.ones((0, 2)), np.ones((1, 2))))
    expect_value_error("批量 QK 批次数不同", lambda: batched_scaled_qk_scores(np.ones((2, 3, 4)), np.ones((3, 3, 4))))
    expect_value_error("批量 QK 向量维度不同", lambda: batched_scaled_qk_scores(np.ones((2, 3, 4)), np.ones((2, 3, 5))))
    expect_value_error("批量 QK 维度错误", lambda: batched_scaled_qk_scores(np.ones((3, 4)), np.ones((3, 4))))

    print(f"通过固定检查：{passed} 项")
    return passed

run_checks()

通过固定检查：35 项


35

## 14. 本课验收与下一课

做完后，请尝试不查答案回答：

1. `[1,2,3]` 在数学意义上是三维向量，为什么 NumPy 中的 `ndim` 却是 `1`？
2. `(2,3) @ (3,4)` 的输出是什么形状？`(2,3) @ (2,3)` 为什么不合法？
3. `(B,T,D) + (D,)` 为什么能广播？
4. 为什么三维的 `K` 不应该直接使用 `.T` 来实现逐 batch 的 `QKᵀ`？
5. 今天计算的 `QKᵀ / √D` 为什么还不能称为注意力**概率**？

**完成标准**：练习 B/C 独立实现并通过检查、能在纸上预测常见形状、看懂 `Q @ K.T` 的每一步。

**第 10 课预告**：softmax、数值稳定性、因果掩码。届时会把分数矩阵转换成注意力权重，再与 `V` 相乘，拼出完整的缩放点积注意力。